Guests by frequency for btw25

With our data from lanz-mining, we can use jupyter to build some simple scripts and explore the obtained data. Further we can show the background and make the process a little more transparent (at least for people with some coding knowledge).

Here we a more narrow time range from the coalition breaking (06.11.24) until 02.02.25. The objective here is to find the vibe of wording in this german elecation campaign.

In [2]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
from datetime import date, datetime
from pathlib import Path

from lanz_mining import params
from lanz_mining.dataproc import text
from lanz_mining.projects.utils import get_all_basic_frames, df_to_time_range

In [6]:
lanz_df, illner_df, miosga_df = get_all_basic_frames(["description", "date", "name", "talkshow"], "../../")
lanz_df.shape, illner_df.shape, miosga_df.shape

((1358, 5), (190, 5), (100, 5))

In [16]:
maisch_df = params.TALKSHOWS["maischberger"]["processor"](
    Path("../../exports/export-maisch.csv")
).dataframe.rename({"maischepisode_name": "episode_name"})
maisch_df = maisch_df["episode_name", "description", "date", "name", "talkshow"]
maisch_df.shape

(414, 5)

In [22]:
# We drop lanz and maischberger since he has no titles
all_df: pl.DataFrame = pl.concat([illner_df, miosga_df], how="vertical")
start = date(2024, 11, 6)
latest = all_df["date"].max()
all_df = df_to_time_range(all_df, start, datetime.today())

In [41]:
titles_df = all_df.group_by("episode_name").agg([pl.col(c).first() for c in ("date", "talkshow")]).sort("date")
titles_df.head(5)

episode_name,date,talkshow
str,datetime[μs],str
"""Beben in Berlin und Washington…",2024-11-07 00:00:00,"""maybritillner"""
"""Wie geht es weiter, Herr Bunde…",2024-11-10 00:00:00,"""carenmiosga"""
"""Ampel weg, Wahlkampf da – hoff…",2024-11-14 00:00:00,"""maybritillner"""
"""Scholz unbeirrbar – Wahlkampf …",2024-11-21 00:00:00,"""maybritillner"""
"""Vor den Neuwahlen – wie grün w…",2024-11-24 00:00:00,"""carenmiosga"""


In [48]:
titles_df["episode_name", "talkshow", "date"].to_numpy()

array([['Beben in Berlin und Washington – wie geht es jetzt weiter?',
        'maybritillner', datetime.datetime(2024, 11, 7, 0, 0)],
       ['Wie geht es weiter, Herr Bundeskanzler?', 'carenmiosga',
        datetime.datetime(2024, 11, 10, 0, 0)],
       ['Ampel weg, Wahlkampf da – hoffen auf einen Neustart?',
        'maybritillner', datetime.datetime(2024, 11, 14, 0, 0)],
       ['Scholz unbeirrbar – Wahlkampf um Krieg und Frieden?',
        'maybritillner', datetime.datetime(2024, 11, 21, 0, 0)],
       ['Vor den Neuwahlen – wie grün wird die Zukunft, Herr Habeck?',
        'carenmiosga', datetime.datetime(2024, 11, 24, 0, 0)],
       ['Wie gut haben Sie regiert, Frau Merkel?', 'maybritillner',
        datetime.datetime(2024, 11, 28, 0, 0)],
       ['Wollten Sie die Wirtschaft oder die FDP retten?', 'carenmiosga',
        datetime.datetime(2024, 12, 1, 0, 0)],
       ['Krieg, Inflation, Abschwung – Deutschland vor der Wahl',
        'maybritillner', datetime.datetime(2024, 12, 5, 0,